In [26]:
%pip install presidio-analyzer presidio-anonymizer spacy
!python3 -m spacy download pt_core_news_lg


Note: you may need to restart the kernel to use updated packages.
  Using cached https://github.com/explosion/spacy-models/releases/download/pt_core_news_lg-3.8.0/pt_core_news_lg-3.8.0-py3-none-any.whl (568.2 MB)
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')


In [2]:
import pandas as pd
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine



In [5]:
# 1. Configura o motor de NLP para usar o modelo em português do spaCy
configuracao_pt = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "pt", "model_name": "pt_core_news_lg"}]
}

# 2. Inicializa os motores do Presidio (Globalmente para manter a performance)
provider = NlpEngineProvider(nlp_configuration=configuracao_pt)
nlp_engine_portugues = provider.create_engine()

analyzer = AnalyzerEngine(nlp_engine=nlp_engine_portugues, supported_languages=["pt"])
anonymizer = AnonymizerEngine()

# 3. Função de Anonimização Adaptada para o DataFrame
def anonimizar_coluna_pt(texto, apenas_entidades=None, ignorar_entidades=None):
    """
    Processa e anonimiza textos em português brasileiro dentro de um DataFrame.
    
    :param apenas_entidades: Lista (ex: ["EMAIL_ADDRESS"]). Se informada, apenas estas serão apagadas.
    :param ignorar_entidades: Lista (ex: ["PERSON"]). Se informada, estas serão preservadas.
    """
    if pd.isna(texto) or not isinstance(texto, str):
        return texto
        
    try:
        # Analisa o texto usando as regras em português
        resultados_analise = analyzer.analyze(
            text=texto, 
            language="pt", 
            entities=apenas_entidades
        )
        
        # Filtra entidades caso o usuário queira ignorar alguma explicitamente
        if ignorar_entidades:
            resultados_analise = [
                res for res in resultados_analise 
                if res.entity_type not in ignorar_entidades
            ]
        
        # Transforma o texto aplicando as máscaras
        resultado_anonimizado = anonymizer.anonymize(
            text=texto, 
            analyzer_results=resultados_analise
        )
        return resultado_anonimizado.text
        
    except Exception as e:
        print(e)
        # Retorna o texto original de forma segura se houver falhas no pipeline
        return texto

In [7]:
df = pd.read_csv("data/smart_objects/smart_objects_questions.csv")

In [9]:
df["question"] = df["question"].apply(lambda x: anonimizar_coluna_pt(x))
df["answer"] = df["answer"].apply(lambda x: anonimizar_coluna_pt(x))

df.to_csv("s_o_q.csv", index=False)